# Sieci Konwolucyjne

In [ ]:
# Wczytanie potrzebnych bibliotek

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # wyciszenie komunikatów TensorFlow

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers
import keras
import kagglehub

from sklearn.metrics import confusion_matrix

---
# 1. Klasyfikacja cyfr – MNIST

**MNIST** to klasyczny zbiór danych zawierający 70 000 odręcznie pisanych cyfr (0–9) w skali szarości o rozmiarze 28×28 pikseli:
- 60 000 obrazów treningowych
- 10 000 obrazów testowych

Zostanie użyta architektura wzorowana na **LeNet-5** – jednej z pierwszych skutecznych sieci konwolucyjnych (LeCun et al., 1998).


In [ ]:
# Wczytanie danych MNIST – keras pobiera je automatycznie przy pierwszym uruchomieniu
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalizacja pikseli do zakresu [0.0, 1.0]
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

# Dodanie wymiaru kanału: (28, 28) → (28, 28, 1)  (Conv2D wymaga wymiaru kanału)
x_train = x_train[..., np.newaxis]
x_test  = x_test[...,  np.newaxis]

print(f"Zbiór treningowy: {x_train.shape},  etykiety: {y_train.shape}")
print(f"Zbiór testowy:    {x_test.shape},  etykiety: {y_test.shape}")
print(f"Liczba klas: {len(np.unique(y_train))}")

In [ ]:
# Wyświetlenie przykładowych obrazów ze zbioru treningowego
fig, axes = plt.subplots(2, 5, figsize=(13, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i].squeeze(), cmap='gray')
    ax.set_title(f'Cyfra: {y_train[i]}', fontsize=13)
    ax.axis('off')
plt.suptitle('Przykładowe obrazy ze zbioru treningowego MNIST', fontsize=14)
plt.tight_layout()
plt.show()

## Architektura sieci – LeNet-5

Sieć LeNet-5 to klasyczna architektura CNN, składa się z:

| Warstwa | Opis |
|---|---|
| `Conv2D(6, 5×5, same)` | Wyodrębnia 6 map cech (krawędzie, kształty), padding=same zachowuje wymiar |
| `AvgPool2D(2)` | Zmniejsza wymiar 2×, dodaje odporność na drobne przesunięcia |
| `Conv2D(16, 5×5, valid)` | Wyodrębnia 16 bardziej złożonych map cech |
| `AvgPool2D(2)` | Kolejna redukcja wymiaru |
| `Flatten` | Spłaszczenie do wektora cech |
| `Dense(120)` + `Dense(84)` | Warstwy klasyfikujące |
| `Dense(10, softmax)` | Wyjście: prawdopodobieństwo każdej z 10 cyfr |

Funkcja straty: `sparse_categorical_crossentropy` – dla etykiet jako liczb całkowitych (nie one-hot).


In [ ]:
# Definicja modelu LeNet-5
model_mnist = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(filters=6,  kernel_size=(5, 5), padding="same",  activation="relu"),
    layers.AvgPool2D(pool_size=2),
    layers.Conv2D(filters=16, kernel_size=(5, 5), padding="valid", activation="relu"),
    layers.AvgPool2D(pool_size=2),
    layers.Flatten(),
    layers.Dense(120, activation="relu"),
    layers.Dense(84,  activation="relu"),
    layers.Dense(10,  activation="softmax"),
], name="LeNet5_MNIST")

model_mnist.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model_mnist.summary()

In [ ]:
# Uczenie modelu – 10 epok, zbiór testowy używany jako walidacyjny
history_mnist = model_mnist.fit(
    x_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(x_test, y_test),
    verbose=1,
)

## Wykresy przebiegu uczenia

- **Dokładność (accuracy)** – procent poprawnie sklasyfikowanych próbek (im wyżej tym lepiej)
- **Funkcja straty (loss)** – miara błędu (im niżej tym lepiej)

Jeśli krzywa walidacyjna znacznie odbiega od treningowej w górę → model się **przeucza (overfitting)**. W takim wypadku należy zastosować regularyzację (Dropout, BatchNormalization, augmentację danych) lub użyć mniej skomplikowanego modelu.


In [ ]:
# Wykresy dokładności i funkcji straty
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_mnist.history['accuracy'],     label='trening',   marker='o')
ax1.plot(history_mnist.history['val_accuracy'], label='walidacja', marker='s')
ax1.set_title('Dokładność klasyfikacji', fontsize=13)
ax1.set_xlabel('Epoka')
ax1.set_ylabel('Dokładność')
ax1.legend()
ax1.grid(True)

ax2.plot(history_mnist.history['loss'],     label='trening',   marker='o')
ax2.plot(history_mnist.history['val_loss'], label='walidacja', marker='s')
ax2.set_title('Funkcja straty', fontsize=13)
ax2.set_xlabel('Epoka')
ax2.set_ylabel('Strata')
ax2.legend()
ax2.grid(True)

plt.suptitle('Przebieg uczenia – MNIST', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Ewaluacja na zbiorze testowym
test_loss, test_acc = model_mnist.evaluate(x_test, y_test, verbose=0)
print(f"Dokładność na zbiorze testowym: {test_acc:.4f}  ({test_acc * 100:.2f}%)")
print(f"Strata na zbiorze testowym:     {test_loss:.4f}")

In [ ]:
# Przykładowe predykcje na zbiorze testowym
n = 10
y_pred_proba = model_mnist.predict(x_test[:n], verbose=0)
y_pred       = y_pred_proba.argmax(axis=1)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_test[i].squeeze(), cmap='gray')
    correct = y_pred[i] == y_test[i]
    color   = 'green' if correct else 'red'
    ax.set_title(f'Pred: {y_pred[i]}  |  Praw: {y_test[i]}', color=color, fontsize=11)
    ax.axis('off')
plt.suptitle('Przykładowe predykcje (zielony = poprawna, czerwony = błędna)', fontsize=13)
plt.tight_layout()
plt.show()

## Macierz pomyłek

**Macierz pomyłek** (confusion matrix) pokazuje, jakie błędy popełnia model:
- **Wiersze** – rzeczywista klasa
- **Kolumny** – przewidywana klasa
- **Przekątna** – poprawne klasyfikacje
- **Poza przekątną** – błędy: np. wartość w wierszu 4, kolumnie 9 oznacza ile razy cyfra 4 została błędnie sklasyfikowana jako 9

Idealna macierz ma wartości niezerowe wyłącznie na głównej przekątnej.


In [ ]:
# Wyznaczenie predykcji dla całego zbioru testowego
y_pred_all = model_mnist.predict(x_test, verbose=0).argmax(axis=1)
cm_mnist   = confusion_matrix(y_test, y_pred_all)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_mnist, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title('Macierz pomyłek – MNIST (zbiór testowy)', fontsize=14)
plt.xlabel('Przewidywana cyfra')
plt.ylabel('Rzeczywista cyfra')
plt.tight_layout()
plt.show()

---
# 2. Klasyfikacja zwierząt – Animals10

Zadanie: klasyfikacja zdjęć kolorowych (RGB, 128×128 px) ze zbioru **Animals10**.

**10 klas:** pies, koń, słoń, motyl, kura, kot, krowa, owca, pająk, wiewiórka

Zbiór zawiera ok. 26 000 zdjęć (oryginalne nazwy klas są po włosku).


In [ ]:
# Pobranie zbioru danych Animals10 przez kagglehub (tylko przy pierwszym uruchomieniu)
path = kagglehub.dataset_download("alessiocorrado99/animals10")
print(f"Dane dostępne pod: {path}")

In [ ]:
# Parametry wczytywania
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

# Słownik tłumaczeń klas (włoski → polski)
translations = {
    "cane":      "pies",
    "cavallo":   "koń",
    "elefante":  "słoń",
    "farfalla":  "motyl",
    "gallina":   "kura",
    "gatto":     "kot",
    "mucca":     "krowa",
    "pecora":    "owca",
    "ragno":     "pająk",
    "scoiattolo":"wiewiórka",
}

# Wczytanie z podziałem 80% trening / 20% walidacja
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(path, "raw-img"),
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    color_mode="rgb", label_mode="int",
    validation_split=0.2, subset="training", seed=42,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(path, "raw-img"),
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    color_mode="rgb", label_mode="int",
    validation_split=0.2, subset="validation", seed=42,
)

class_names    = train_ds.class_names
class_names_pl = [translations.get(c, c) for c in class_names]

print("Klasy:", class_names_pl)
print(f"Liczba batchy treningowych:  {len(train_ds)}")
print(f"Liczba batchy walidacyjnych: {len(val_ds)}")

In [ ]:
# Wyświetlenie przykładowych obrazów ze zbioru treningowego
images, labels = next(iter(train_ds))

fig, axes = plt.subplots(3, 6, figsize=(16, 9))
for i, ax in enumerate(axes.flat):
    if i < len(images):
        ax.imshow(images[i].numpy().astype('uint8'))
        ax.set_title(class_names_pl[labels[i].numpy()], fontsize=11)
    ax.axis('off')
plt.suptitle('Przykładowe obrazy ze zbioru treningowego Animals10', fontsize=14)
plt.tight_layout()
plt.show()

## Zadanie: implementacja sieci konwolucyjnej dla Animals10

Należy zaimplementować sieć konwolucyjną (CNN) do klasyfikacji kolorowych zdjęć zwierząt (10 klas, obrazy RGB 128×128 px).

Sieć powinna składać się z bloków konwolucyjnych zakończonych warstwami klasyfikującymi. Poniżej zestawiono warstwy dostępne w Keras wraz z ich zastosowaniem:

| Warstwa | Zastosowanie |
|---|---|
| `Conv2D(filters, kernel_size, padding, activation)` | Wyodrębnia lokalne cechy przestrzenne (krawędzie, tekstury, kształty) |
| `MaxPooling2D(pool_size)` | Redukuje wymiar przestrzenny, zachowuje dominujące cechy |
| `AveragePooling2D(pool_size)` | Redukuje wymiar przez uśrednianie – łagodniejsze niż MaxPool |
| `GlobalAveragePooling2D()` | Agreguje całą mapę cech do jednej wartości – alternatywa dla Flatten |
| `Flatten()` | Spłaszcza mapy cech do wektora przed warstwami gęstymi |
| `Dense(units, activation)` | Warstwa w pełni połączona – klasyfikacja na podstawie wyodrębnionych cech |
| `Dropout(rate)` | Losowo wyłącza neurony podczas treningu – ogranicza przeuczenie |
| `BatchNormalization()` | Normalizuje aktywacje w obrębie batcha – stabilizuje i przyspiesza uczenie |
| `Rescaling(1/255)` | Normalizuje piksele z zakresu [0, 255] do [0, 1] wewnątrz modelu |

Sieć należy skompilować z odpowiednią funkcją straty i metryką, a następnie wytrenować na zbiorze treningowym z walidacją na zbiorze walidacyjnym.


In [ ]:
model_animals = keras.Sequential([
    # TODO: Uzupełnij model sieci
], name="CNN_Animals10")

model_animals.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model_animals.summary()


In [ ]:
# Uczenie modelu – może trwać kilka minut
history_animals = model_animals.fit(
    train_ds,
    epochs=15,
    validation_data=val_ds,
    verbose=1,
)

In [ ]:
# Wykresy dokładności i funkcji straty
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_animals.history['accuracy'],     label='trening',   marker='o')
ax1.plot(history_animals.history['val_accuracy'], label='walidacja', marker='s')
ax1.set_title('Dokładność klasyfikacji', fontsize=13)
ax1.set_xlabel('Epoka')
ax1.set_ylabel('Dokładność')
ax1.legend()
ax1.grid(True)

ax2.plot(history_animals.history['loss'],     label='trening',   marker='o')
ax2.plot(history_animals.history['val_loss'], label='walidacja', marker='s')
ax2.set_title('Funkcja straty', fontsize=13)
ax2.set_xlabel('Epoka')
ax2.set_ylabel('Strata')
ax2.legend()
ax2.grid(True)

plt.suptitle('Przebieg uczenia – Animals10', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Ewaluacja na zbiorze walidacyjnym
val_loss, val_acc = model_animals.evaluate(val_ds, verbose=0)
print(f"Dokładność na zbiorze walidacyjnym: {val_acc:.4f}  ({val_acc * 100:.2f}%)")
print(f"Strata na zbiorze walidacyjnym:     {val_loss:.4f}")

In [ ]:
# Przykładowe predykcje na zbiorze walidacyjnym
images_val, labels_val = next(iter(val_ds))
images_sample = images_val[:9]
labels_sample = labels_val.numpy()[:9]

preds_sample  = model_animals.predict(images_sample, verbose=0)
pred_classes  = preds_sample.argmax(axis=1)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for i, ax in enumerate(axes.flat):
    ax.imshow(images_sample[i].numpy().astype('uint8'))
    correct = pred_classes[i] == labels_sample[i]
    color   = 'green' if correct else 'red'
    ax.set_title(
        f'Pred: {class_names_pl[pred_classes[i]]}\nPraw: {class_names_pl[labels_sample[i]]}',
        color=color, fontsize=11,
    )
    ax.axis('off')
plt.suptitle('Przykładowe predykcje – Animals10\n(zielony = poprawna, czerwony = błędna)', fontsize=13)
plt.tight_layout()
plt.show()

## Macierz pomyłek – Animals10

Macierz pomyłek pozwala zidentyfikować, które klasy zwierząt są najczęściej mylone ze sobą. Można zaobserwować np. czy model myli wizualnie podobne zwierzęta (np. krowa ↔ koń, kura ↔ motyl).


In [ ]:
# Zebranie wszystkich predykcji i etykiet ze zbioru walidacyjnego
y_true_a, y_pred_a = [], []
for imgs_b, lbl_b in val_ds:
    p = model_animals.predict(imgs_b, verbose=0)
    y_pred_a.extend(p.argmax(axis=1).tolist())
    y_true_a.extend(lbl_b.numpy().tolist())

cm_animals = confusion_matrix(y_true_a, y_pred_a)

plt.figure(figsize=(12, 10))
sns.heatmap(cm_animals, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names_pl, yticklabels=class_names_pl)
plt.title('Macierz pomyłek – Animals10 (zbiór walidacyjny)', fontsize=14)
plt.xlabel('Przewidywana klasa')
plt.ylabel('Rzeczywista klasa')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

Pytania:
1. Czy udało się w pełni zapobiec przeuczeniu modelu do danych treningowych?
2. Które zwierzęta są najczęściej mylone z którymi?
3. Jak learning rate wpływa na prędkość i stabilność uczenia?